# Lighthouse Notebook 2: zero shot classification

Each of you is building a different model this week on the same data. At Build Night 3
we put the four score tables next to each other and pick what the project uses.

The data is different from last week. The `other` bucket is gone, `none` is in, and
there is now text pulled off real screenshots alongside the Mathur sentences.

The setup cells load the data. How you get from there to a working model is up to you.
You have AI tools available. Use them to write code, not to decide what to do.

Turn on the GPU before you start: Runtime, Change runtime type, T4 GPU.

## Setup

Run these three. Upload `contextdp_ocr.csv` from the shared folder first, using the file
browser on the left.

In [4]:
URL = ("https://raw.githubusercontent.com/aruneshmathur/dark-patterns/"
       "master/data/final-dark-patterns/dark-patterns.csv")

MATHUR_TO_OURS = {
    "Low-stock Message":                "scarcity",
    "High-demand Message":              "scarcity",
    "Countdown Timer":                  "urgency",
    "Limited-time Message":             "urgency",
    "Activity Notification":            "social_proof",
    "Testimonials of Uncertain Origin": "social_proof",
    "Confirmshaming":                   "confirmshaming",
    "Visual Interference":              "hidden_information",
}

import pandas as pd, numpy as np

mathur = pd.read_csv(URL)
mathur["label"] = mathur["Pattern Type"].map(MATHUR_TO_OURS)
mathur = mathur.dropna(subset=["label", "Pattern String"]).copy()
mathur["text"] = mathur["Pattern String"].str.strip()
mathur = mathur[mathur["text"] != ""].drop_duplicates("text")
mathur["site"] = (mathur["Website Page"].astype(str)
                  .str.extract(r"https?://([^/]+)")[0]
                  .str.replace(r"^www\.", "", regex=True))
mathur = mathur.dropna(subset=["site"])[["text", "label", "site"]].reset_index(drop=True)

print(len(mathur), "sentences from", mathur["site"].nunique(), "websites")
print(mathur["label"].value_counts())

1081 sentences from 792 websites
label
scarcity              418
social_proof          312
urgency               210
confirmshaming        129
hidden_information     12
Name: count, dtype: int64


In [5]:
# 440 screenshots from ContextDP, labelled with the mapping we built at Build Night 2,
# with the text already pulled off them by OCR.

shots = pd.read_csv("contextdp_ocr.csv")
print(len(shots), "screenshots")
print(shots["label"].value_counts())
shots.head(3)

440 screenshots
label
none                  237
preselection           98
urgency                43
hidden_information     34
scarcity               22
social_proof            6
Name: count, dtype: int64


,image,platform,their_labels,label,text
0,droom.in.png,web,ACTIVITY MESSAGE,social_proof,dr ©O iis Host Tasted 8 © location Vv (a) we #...
1,kiiroo.com.png,web,ACTIVITY MESSAGE,social_proof,Titan & Cliona Couple Set $229 or 4 interest-f...
2,nutrabay.com.png,web,ACTIVITY MESSAGE,social_proof,NUTRABAY BRANDS SPORTS NUTRITION WEIGHT Loss W...


In [6]:
held_out = pd.DataFrame([
    ("Better grab it, barely any left", "scarcity"),
    ("Buy it before the timer runs out", "urgency"),
    ("There are hardly any of these remaining", "scarcity"),
    ("You would be missing out if you say no", "confirmshaming"),
    ("Everyone is buying this right now", "social_proof"),
    ("People can't stop buying this product", "social_proof"),
    ("The performance of the product is better now, only 17 made ever", "scarcity"),
    ("Are you sure you want to stop protecting your family?", "confirmshaming"),
    ("not a dark pattern, nothing to see here", "none"),
    ("1000 left", "none"),
    ("50% but only for the next decade", "none"),
    ("between 0 and 1 million left in stock", "none"),
    ("20% off but just for today", "urgency"),
    ("Sale ends in 01:59:00", "urgency"),
    ("bought 200 times, only ten left in stock", "scarcity"),
    ("bought 200 times", "social_proof"),
    ("Limited time offer: get 30% off your first 3 months till september 30th. "
     "You can always cancel anytime", "urgency"),
    ("10 days left for our end of summer sale. Get 50% off until the end of the month.",
     "urgency"),
    ("Get our new student discount. 15% off with a valid student ID", "none"),
    ("Rate our Product! Remind me later.", "none"),
    ("Toggled settings to allow notifications and data collection by default",
     "preselection"),
    ("90k people have viewed this item", "social_proof"),
    ("35% off on all items for a limited time only!", "urgency"),
    ("only 1 item left in stock", "scarcity"),
    ("25 people have this in their cart at the moment", "scarcity"),
    ("No thanks, I'd rather pay full price", "confirmshaming"),
    ("Over 12,000 people bought this today", "social_proof"),
    ("Student discounts offered", "none"),
    ("70% of our users are making more than 700K", "social_proof"),
    ("Everyone in dallas is buying the new ford bronco, now is your time", "social_proof"),
    ("The buy one get one free deal will end in 3 hours, shop now", "urgency"),
    ("the new harry potter movie is dropping on 22 september this year", "none"),
    ("Free shipping on orders over $50", "none"),
    ("Currently out of stock", "none"),
    ("4.6 out of 5 stars from 1,204 reviews", "none"),
    ("Summer sale: 20% off all sandals", "none"),
    ("Ships in 3 to 5 business days", "none"),
    ("Return it within 30 days for a full refund", "none"),
    ("Now available in three colors", "none"),
    ("Add to cart", "none"),
    ("In stock at your local store", "none"),
    ("Save this item to your wishlist", "none"),
    ("We use cookies. Accept all or manage your preferences.", "none"),
    ("Compare similar items", "none"),
    ("Price: $24.99, tax calculated at checkout", "none"),
    ("Sign in to see your order history", "none"),
    ("This product weighs 2.3 lbs and ships from Ohio", "none"),
], columns=["text", "label"])

print(len(held_out), "held out sentences,", (held_out.label == "none").sum(), "of them none")

47 held out sentences, 23 of them none


## 1. Split the data

Use these exact settings. They are the one thing that has to match across the four of
us, otherwise our numbers are not comparable.

Sentences: grouped split on the `site` column, 20% test, `random_state=42`. Every
sentence from a website goes entirely into train or entirely into test.

Screenshots: 60/40 split, stratified on label, `random_state=42`.

Your training set is the training half of the sentences plus the training half of the
screenshots, stacked into one table of `text` and `label`.

In [7]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
import pandas as pd

gss= GroupShuffleSplit (n_splits=1, test_size=0.2, random_state=42)
#you are making n split to 1 because you are splitting the data only one time
#for test and train but you can split in more pieces(not in this one)
#we are putting only 20 percent of the data in the test category so when
#the websites will be moved we can just get the 20 percent in that process as well

train_idx, test_idx = next(gss.split(mathur, groups=mathur["site"]))
#this is where we are splitting the data
#we are defining the site table in here this can be taken as filtering
#the gss function will not return the value it will return the row position since thats
#how the filtering is done

sent_train, sent_test = mathur.iloc[train_idx], mathur.iloc[test_idx]
#this can be seen as the call function of the rows we had the position at first but
#here we are getting the actual rows

shot_train, shot_test = train_test_split(shots, test_size=0.4, stratify=shots["label"], random_state=42)
#here we are spliiting the screenshots of the sites we did sentences first and now this is for the screenshots
#40 percent for the test ans the rest for the training
#the test train split is going to split the whole data equally in the test and train
#meaning after splitting the 60 of the training will have the same ratios of the category as in the
#40 percent of the test

train = pd.concat([sent_train[["text","label"]], shot_train[["text","label"]]]). reset_index(drop=True)
#in this we are only keeping the text and th label category in the site and dropping the other categories





## 2. Zero shot classification

Your model does no training at all. You hand it our category descriptions and it decides
which one a sentence matches. This is the closest thing to what we do with the vision
model next week, so your results tell us whether prompting is worth it before we build
on it.

`facebook/bart-large-mnli` through the `transformers` zero shot classification pipeline
is the straightforward way in. It is about 1.6 GB and runs on the T4.

The real work is the candidate labels you hand it. "a fake countdown or time limit" will
behave very differently from "urgency". Finding wording that works is the task, so keep
notes on what you tried and what changed.

Report your three scores like everyone else even though you never trained on anything.

In [8]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report
#here we are loading the modelas and running and we are also importing
#accuracy score and the classification report

clf = pipeline("zero-shot-classification",
               model="facebook/bart-large-mnli",
               device=0)
# in the zero shot classification we are checking if is telling the pipleine
#for is the sentences related to a label and we are running the model of the GPU

DESCRIPTIONS = {
    "claiming that only a few items are left":           "scarcity",
    "there is a limited time to buy the item":            "urgency",
    "the item is popular/ lots of people are viewing and buying it":       "social_proof",
    "Guilt driving you to buy the item when you say no":     "confirmshaming",
    "Framing effect or hiding something that you might want to use": "hidden_information",
    "no dark pattern detected":               "none",
}


def predict(texts):
    results = clf(list(texts), candidate_labels=list(DESCRIPTIONS), batch_size=16)
    return [DESCRIPTIONS[r["labels"][0]] for r in results]

#here we are taking the text in the plain list for python for the pipeline
#we will get the keys
# we will do it in a batch of 16 sentences


def score(name, df):
    preds = predict(df["text"])
    print(f"{name}: accuracy {accuracy_score(df['label'], preds):.3f}")
    print(classification_report(df["label"], preds, zero_division=0))

# this is used for the accuracy score and till 3 decimal spaces






config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## 3. Look at the OCR text

Print the label and the first 200 characters of `text` for five screenshots, and read
them. You need this for question 3.

In [9]:
for _, row in shots.head(5).iterrows():
    print("LABEL:", row["label"])
    print(row["text"][:200])
    print("-" * 60)
#_ is that the for loop doesnot need the value to work on it

LABEL: social_proof
dr ©O iis Host Tasted 8 © location Vv (a) we #5 Categories Buy Sell/Exchange Vehicle Finder. How Droom Works @WatchTVC Home / Bikes / New / Honda CB Hornet 1608 / Honda CB Hornet 160R ABS DLX 2019 © H
------------------------------------------------------------
LABEL: social_proof
Titan & Cliona Couple Set $229 or 4 interest-free payments of $57.25 with @ sezzle @ FREE SHIPPING. Discreet Secure Achievements ‘Over $100 packaging checkout Add to cart 9 AddtoV 95 users viewed this
------------------------------------------------------------
LABEL: social_proof
NUTRABAY BRANDS SPORTS NUTRITION WEIGHT Loss WELLNESS PROTEIN POWDERS VITAMINS CRAZY DEALS TOP 10 LOGIN/REGISTER CART/70 a) jonmx% 7 Giocovare creme Ultimate Nutrition Ultimate Nutrition Prostar 100 W
------------------------------------------------------------
LABEL: urgency
Clothing, Giftboxes Qs usD+ = CART HOME JEWELRY ¥ APPAREL ¥ EDUCATION ¥ ABOUTUS FAQ SUPPORT & LIMITED EDITION HONEYCOMB HEART NECKLACE (BEE

## 4. Score it three ways

Print a `classification_report` for each:

1. the sentence test set
2. the screenshot test set
3. `held_out`

In [10]:
score("test on the mathurs sentences", sent_test)
score("test on the screenshot we collected", shot_test)
score("something that is different in the data(HELD_OUT)", held_out)

test on the mathurs sentences: accuracy 0.601
                    precision    recall  f1-score   support

    confirmshaming       0.00      0.00      0.00        28
hidden_information       0.00      0.00      0.00         1
              none       0.00      0.00      0.00         0
          scarcity       0.88      0.69      0.77        84
      social_proof       0.86      0.54      0.67        68
           urgency       0.60      0.97      0.74        37

          accuracy                           0.60       218
         macro avg       0.39      0.37      0.36       218
      weighted avg       0.71      0.60      0.63       218

test on the screenshot we collected: accuracy 0.142
                    precision    recall  f1-score   support

hidden_information       0.10      0.50      0.17        14
              none       1.00      0.02      0.04        95
      preselection       0.00      0.00      0.00        39
          scarcity       0.07      0.33      0.12         

## 5. Find what it got wrong

Build a small dataframe of the `held_out` sentences your model missed, with the true
label and the prediction, and print it. Bring this to Build Night 3.

In [11]:
pd.set_option("display.max_colwidth", None)
#by this pandas will be giving whole sentences instead of cutting it off
missed = held_out.copy()
missed["prediction"] = predict(held_out["text"])
missed = missed[missed["label"] != missed["prediction"]]
# here we are going over the held out sentences and getting the models prediction

print(len(missed), "missed out of", len(held_out))
print(missed)

29 missed out of 47
                                                                      text  \
3                                   You would be missing out if you say no   
7                    Are you sure you want to stop protecting your family?   
9                                                                1000 left   
10                                        50% but only for the next decade   
11                                   between 0 and 1 million left in stock   
18           Get our new student discount. 15% off with a valid student ID   
19                                      Rate our Product! Remind me later.   
20  Toggled settings to allow notifications and data collection by default   
23                                               only 1 item left in stock   
24                         25 people have this in their cart at the moment   
25                                    No thanks, I'd rather pay full price   
27                                          

## 6. Save three files

Use these names and columns so they merge:

`results_YOURNAME.csv` with columns `model, eval_set, accuracy, macro_f1`, one row per
eval set. Model name for this notebook is `zero_shot_bart`.

`wrong_YOURNAME.csv`, the dataframe from step 5.

`preds_YOURNAME.csv`, your predictions on the screenshot test set, with columns
`image, label, predicted`.

Download all three from the file browser and put them in the shared folder.

In [12]:
from sklearn.metrics import f1_score

NAME = "Muhammad Bilal"


eval_sets = {
    "sentences":   sent_test,
    "screenshots": shot_test,
    "held_out":    held_out,
}

rows = []
all_preds = {}
for set_name, df in eval_sets.items():
    preds = predict(df["text"])
    all_preds[set_name] = preds
    rows.append({
        "model":    "zero_shot_bart",
        "eval_set": set_name,
        "accuracy": accuracy_score(df["label"], preds),
        "macro_f1": f1_score(df["label"], preds, average="macro", zero_division=0),
    })

results = pd.DataFrame(rows)
results.to_csv(f"results_{NAME}.csv", index=False)
print(results)


missed.to_csv(f"wrong_{NAME}.csv", index=False)

preds_df = shot_test[["image", "label"]].copy()
preds_df["predicted"] = all_preds["screenshots"]
preds_df.to_csv(f"preds_{NAME}.csv", index=False)

print("Saved all three files")

            model     eval_set  accuracy  macro_f1
0  zero_shot_bart    sentences  0.600917  0.363711
1  zero_shot_bart  screenshots  0.142045  0.156163
2  zero_shot_bart     held_out  0.382979  0.283766
Saved all three files


## 7. Questions

1. Your three accuracy numbers.
2. Why is the screenshot score different from the sentence score?
3. How is OCR text different from a Mathur sentence? One line.
4. Two held out sentences you got wrong, with what the model guessed. Was one of them a
   `none`?
5. Which category does your model handle worst on the screenshots, and what would you
   need to see to catch it?
6. One thing that confused you.
7. Guess how your model does against the other three. We check next week.

## Submitting

1. Runtime, then Restart session and run all. Wait for it to finish.
2. Scroll through and check every cell has its output showing. If a cell is empty, your
   results did not save.
3. File, Download, Download .ipynb.
4. Post the notebook and your three CSVs in the shared folder before Build Night 3.

If something crashes halfway, fix it and run all again. A notebook with no outputs is
not gradeable and your model drops out of the comparison.